In [3]:
from pgmpy.inference import VariableElimination
import pandas as pd
import numpy as np
from distributed.itertools import ffill
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from sympy.physics.control.control_plots import plt

# time
#
# glucose
#
# calories
#
# heart_rate
#
# steps
#
# basal_rate
#
# bolus_volume_delivered
#
# carb_input
df = pd.read_csv('../data/HUPA0001P.csv', sep=';', parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)

df['time'] = pd.to_datetime(df['time'])

df['carb_input'] = df['carb_input'].fillna(0.0)

# transfer to hour
df['TC_Hour'] = df['time'].dt.hour

df['glucose'] = df['glucose'].ffill().bfill()

df["GL_State"] = pd.cut(df['glucose'], bins=[-1, 70, 140, 999], labels=[1, 2, 3]).astype(int)

df['last_meal_time'] = df.apply(lambda row: row['time'] if row['carb_input'] > 0 else pd.NaT, axis=1)
df['last_meal_time'] = df['last_meal_time'].ffill()
df['hours_since_last'] = (df['time'] - df['last_meal_time']).dt.total_seconds() / 3600
df['hours_since_last'] = df['hours_since_last'].fillna(999)

#df

df['TP_State'] = pd.cut(df['hours_since_last'], bins=[-1, 2, 4, 999], labels=[1, 2, 3]).astype(int)
df['next_meal_time'] = df.apply(lambda row: row['time'] if row['carb_input'] > 0 else pd.NaT, axis=1)
df['next_meal_amount'] = df.apply(lambda row: row['carb_input'] if row['carb_input'] > 0 else pd.NA, axis=1)
df['next_meal_time'] = df['next_meal_time'].bfill()
df['next_meal_amount']=df['next_meal_amount'].bfill()
df['hours_until_next'] = (df['next_meal_time'] - df['time']).dt.total_seconds() / 3600

train_df = df.dropna(subset=['hours_until_next']).copy()
#train_df

train_df['TUN_State'] = pd.cut(train_df['hours_until_next'], bins=[-0.1, 1, 3, 999], labels=[1, 2, 3]).astype(int)
train_df['SN_State'] = pd.cut(train_df['next_meal_amount'], bins=[0, 3, 999], labels=[1, 2]).astype(int)
# (Time current:TC, hours_since_last_past: TP, Glucose level: GL) -----> (Time until next:TUN)
#                  ---->   (Size of next: SN)
model = DiscreteBayesianNetwork([
    ('TC_Hour', 'TUN_State'),
    ('GL_State', 'TUN_State'),
    ('TP_State', 'TUN_State'),
    ('TUN_State', 'SN_State'),
])
train_cols = ['TC_Hour', 'GL_State', 'TP_State', 'TUN_State', 'SN_State']
model.fit(
    train_df[train_cols],
    #estimator=BayesianEstimator
    estimator=MaximumLikelihoodEstimator
)

print("training down!\n")


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'TC_Hour': 'N', 'GL_State': 'N', 'TP_State': 'N', 'TUN_State': 'N', 'SN_State': 'N'}


training down!



In [4]:

infer = VariableElimination(model)

def predict_next_meal(current_hour, current_glucose, hours_since_last_meal):
    print(f"--- evidence ---")
    print(f"current time(hour): {current_hour}")
    print(f"current glucose: {current_glucose}")
    print(f"hours since last meal(hours): {hours_since_last_meal}\n")

    gl_state = 1 if current_glucose < 70 else (2 if current_glucose <= 140 else 3)
    tp_state = 1 if hours_since_last_meal < 2 else (2 if hours_since_last_meal <= 4 else 3)

    evidence_dict = {
        'TC_Hour': current_hour,
        'GL_State': gl_state,
        'TP_State': tp_state
    }

    # TUN_State
    print(">>> Prediction TUN State (1: within one hours, 2:1-3 hours, 3:above 3 hours)")
    res_tun_state = infer.query(variables=['TUN_State'], evidence=evidence_dict)
    print(res_tun_state)

    # SN_State
    print(">>> Prediction SN State (1: <=3 means snacks, 2: >3 means normal meal")
    res_sn_state = infer.query(variables=['SN_State'], evidence=evidence_dict)
    print(res_sn_state)


# test1
predict_next_meal(current_hour=14, current_glucose=105, hours_since_last_meal=4.5)

print("-" * 50)

predict_next_meal(current_hour=21, current_glucose=65, hours_since_last_meal=2.5)

--- evidence ---
current time(hour): 14
current glucose: 105
hours since last meal(hours): 4.5

>>> Prediction TUN State (1: within one hours, 2:1-3 hours, 3:above 3 hours)
+--------------+------------------+
| TUN_State    |   phi(TUN_State) |
+==============+==================+
| TUN_State(1) |           0.8000 |
+--------------+------------------+
| TUN_State(2) |           0.0500 |
+--------------+------------------+
| TUN_State(3) |           0.1500 |
+--------------+------------------+
>>> Prediction SN State (1: <=3 means snacks, 2: >3 means normal meal
+-------------+-----------------+
| SN_State    |   phi(SN_State) |
+=============+=================+
| SN_State(1) |          0.6374 |
+-------------+-----------------+
| SN_State(2) |          0.3626 |
+-------------+-----------------+
--------------------------------------------------
--- evidence ---
current time(hour): 21
current glucose: 65
hours since last meal(hours): 2.5

>>> Prediction TUN State (1: within one hours, 2: